# MagicBox training on TPU
Run each stage directly in the notebook kernel. Set `RUN_MODE` to `smoke` for 10 updates or `full` for the configured epochs. Source is loaded from the pinned GitHub checkout. The model reads every question, its answer options, and the text in one encoder pass, then answers each question by pointing at an option or a text span.


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

RUN_MODE = 'smoke'  # 'smoke' or 'full'.
DEVICES = None  # Detect all visible TPUs; set 1 or 8 to require that exact count.
REQUESTS_PER_DEVICE = 1
SMOKE_STEPS = 10
EPOCHS = 3
SESSION_HOURS = 1 if RUN_MODE == 'smoke' else 8
MICROBATCHES = 1 if RUN_MODE == 'smoke' else 4
SEQUENCE_TOKENS = None  # Joint question-and-text tokens per request; None measures every split.
QUESTIONS = None  # Questions per request; None measures every split.
SCORE_WIDTH = 0.15  # Spread hard score labels over nearby levels; 0 keeps them one-hot.
OPTIMIZER = 'optax'  # 'optax' skips non-finite steps cheaply; 'transactional' is the older checked commit.
KEEP_CHECKPOINTS = 2  # Newest checkpoints kept in OUTPUT (about 4.3GB each).
LEARNING_RATE = 2e-5
VALIDATION_RECORDS = 8 if RUN_MODE == 'smoke' else 256
FINAL_RECORDS = 8 if RUN_MODE == 'smoke' else 0

IS_KAGGLE = Path('/kaggle/working').is_dir()
HOST = Path('/kaggle/working') if IS_KAGGLE else Path('/content')
SCRATCH = Path('/kaggle/temp/magicbox') if IS_KAGGLE else HOST / 'magicbox-scratch'
OUTPUT = None  # Defaults to magicbox-{RUN_MODE}-{DEVICES}dev under HOST.
RESUME_FROM = None  # Previous output folder with the same hardware and batch settings.
DATASET = None  # Override with a local completed dataset directory to skip its download.
DATASET_REPO = 'protodotdesign/magicbox-v1'
DATASET_REVISION = 'f074bb549f16ea091fd8ece12e79652b8082871f'
XLA_DUMP = None  # Set a directory to save text HLO for every compiled program.

if OPTIMIZER not in ('optax', 'transactional'):
    raise ValueError("OPTIMIZER must be 'optax' or 'transactional'")
if RUN_MODE not in ('smoke', 'full'):
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")
for name in (
    'REQUESTS_PER_DEVICE', 'MICROBATCHES', 'SMOKE_STEPS',
):
    if type(globals()[name]) is not int or globals()[name] < 1:
        raise ValueError(f'{name} must be a positive integer')
for name in ('SEQUENCE_TOKENS', 'QUESTIONS'):
    if globals()[name] is not None and (type(globals()[name]) is not int or globals()[name] < 1):
        raise ValueError(f'{name} must be None or a positive integer')
SCRATCH.mkdir(parents=True, exist_ok=True)
if XLA_DUMP is not None:
    os.environ['XLA_FLAGS'] = f'--xla_dump_to={XLA_DUMP} --xla_dump_hlo_as_text'
CHECKOUT = SCRATCH / 'minifield-training'
SOURCE_REPO_URL = 'https://github.com/Minifield-Labs/minifield-training.git'
SOURCE_REVISION = '7945e545a712d6a196bd08fd9821fd30d06e128d'
MODEL = SCRATCH / 'encoder'
CACHE = SCRATCH / 'arrow'


SEED = 20260927
RUN_ID = 'magicbox-lfm350m-v1'

def setup_command(*command, cwd=None):
    result = subprocess.run(command, cwd=cwd or CHECKOUT, check=True,
                            text=True, stdout=subprocess.PIPE)
    print(result.stdout, end='', flush=True)
    return result.stdout

## Fetch the pinned source


In [ ]:
if not (CHECKOUT / '.git').is_dir():
    if CHECKOUT.exists():
        raise RuntimeError(f'Expected a clean checkout path: {CHECKOUT}')
    setup_command('git', 'clone', '--no-checkout', SOURCE_REPO_URL, str(CHECKOUT),
              cwd=SCRATCH)
else:
    if setup_command('git', 'status', '--porcelain').strip():
        raise RuntimeError(f'Checkout contains local changes: {CHECKOUT}')
contains_pin = subprocess.run(
    ['git', 'cat-file', '-e', f'{SOURCE_REVISION}^{{commit}}'],
    cwd=CHECKOUT, capture_output=True, check=False).returncode == 0
if not contains_pin:
    setup_command('git', 'fetch', '--depth', '1', SOURCE_REPO_URL, SOURCE_REVISION)
setup_command('git', 'checkout', '--detach', SOURCE_REVISION)
actual_revision = setup_command('git', 'rev-parse', 'HEAD').strip()
if actual_revision != SOURCE_REVISION:
    raise RuntimeError(f'Expected {SOURCE_REVISION}, got {actual_revision}')
if not (CHECKOUT / 'examples/magicbox/train.py').is_file():
    raise RuntimeError('The pinned checkout is missing the MagicBox trainer')
print('Training source:', SOURCE_REPO_URL, actual_revision)

## Install into this kernel


In [ ]:
if not (3, 12) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError('Select a Python 3.12 or 3.13 notebook runtime')
if any(name in sys.modules for name in ('jax', 'jaxlib', 'minifield_training')):
    raise RuntimeError('Restart the notebook session before installing dependencies')
import importlib.metadata
import re
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'uv==0.11.30'])
locked = SCRATCH / 'notebook-requirements.txt'
subprocess.check_call([
    sys.executable, '-m', 'uv', 'export', '--project', str(CHECKOUT),
    '--locked', '--no-dev', '--all-extras', '--no-emit-project', '--no-hashes',
    '--format', 'requirements-txt', '--output-file', str(locked),
])
# Kaggle imports some packages (numpy) before the first cell. Replacing a loaded
# package on disk would need a restart, so keep what the kernel already loaded and
# let pip confirm those versions satisfy everything else.
WATCHED = ('numpy', 'scipy', 'pandas', 'pyarrow', 'transformers', 'tokenizers')
preloaded = {name: sys.modules[name].__version__ for name in WATCHED if name in sys.modules}
requirements = SCRATCH / 'notebook-requirements-kept.txt'
requirements.write_text(''.join(
    line + '\n' for line in locked.read_text().splitlines()
    if re.split(r'[=<>!~; ]', line.strip(), maxsplit=1)[0].lower() not in preloaded
))
constraints = SCRATCH / 'notebook-preloaded.txt'
constraints.write_text(''.join(f'{name}=={version}\n' for name, version in preloaded.items()))
print('Keeping preloaded packages:', preloaded or 'none', flush=True)
extra = [str(CHECKOUT), 'jax[tpu]==0.7.2']
pip = [sys.executable, '-m', 'pip', 'install']
if subprocess.run([*pip, '-r', str(requirements), '-c', str(constraints), *extra]).returncode:
    # Incompatible preloaded versions: install the lock so the restarted kernel loads it.
    subprocess.check_call([*pip, '-r', str(locked), *extra])
    raise RuntimeError('Preloaded packages were incompatible and have been replaced. Restart the session, then run from the first cell.')
for package in WATCHED:
    loaded = sys.modules.get(package)
    if loaded is not None and getattr(loaded, '__version__', None) != importlib.metadata.version(package):
        raise RuntimeError(f'{package} changed on disk. Restart the session, then run from the first cell.')
sys.path.insert(0, str(CHECKOUT))
print('Installed into notebook kernel:', sys.executable)

## Discover TPU devices


In [ ]:
import json
from pathlib import Path
import sys
import jax

devices = jax.devices()
if jax.__version__ != '0.7.2':
    raise RuntimeError('Expected pinned JAX 0.7.2')
if jax.process_count() != 1 or len(devices) != jax.local_device_count():
    raise RuntimeError('This notebook requires all TPU devices on one host')
if not devices or any(device.platform != 'tpu' for device in devices):
    raise RuntimeError('Select a TPU runtime')
hardware = {'devices': len(devices), 'kinds': [d.device_kind for d in devices]}

print(hardware, flush=True)
if DEVICES is None:
    DEVICES = hardware['devices']
if type(DEVICES) is not int or DEVICES < 1 or hardware['devices'] != DEVICES:
    raise ValueError(f"Requested {DEVICES} devices; runtime exposes {hardware['devices']}")
REQUESTS = DEVICES * REQUESTS_PER_DEVICE
if OUTPUT is None:
    OUTPUT = HOST / f'magicbox-{RUN_MODE}-{DEVICES}dev'
OUTPUT = Path(OUTPUT)
print('Mode:', RUN_MODE, 'Devices:', DEVICES,
      'Requests per device:', REQUESTS_PER_DEVICE,
      'Logical batch:', REQUESTS * MICROBATCHES, 'Output:', OUTPUT)

## Download dataset and encoder


In [ ]:
from huggingface_hub import snapshot_download
from minifield_training.models.lfm2_5 import encoder
import shutil

if DATASET is None:
    DATASET = SCRATCH / 'dataset'
    snapshot_download(DATASET_REPO, repo_type='dataset', revision=DATASET_REVISION,
                      local_dir=str(DATASET), allow_patterns=[
                          'manifest.json', 'identity.json', 'report.json', 'sources.json',
                          'tokenizer/*', 'train/*.parquet', 'validation/*.parquet',
                          'calibration/*.parquet', 'test/*.parquet', 'ood/*.parquet'])
DATASET = Path(DATASET)
if RESUME_FROM is not None and not OUTPUT.exists():
    shutil.copytree(RESUME_FROM, OUTPUT)
manifest = json.loads((DATASET / 'manifest.json').read_text())
assert manifest['format'] == 'minifield.magicbox/1.0' and manifest['complete'] is True
assert manifest['mode'] == 'full'
print('Dataset token limits:', manifest['tokenizer']['source_limit'], 'source,',
      manifest['tokenizer']['schema_limit'], 'per schema row')
by_split = {}
for shard in manifest['shards']:
    by_split[shard['split']] = by_split.get(shard['split'], 0) + shard['rows']
print('Dataset:', DATASET, '\nRecords:', by_split)
print('Logical updates per epoch:', (by_split['train'] + REQUESTS * MICROBATCHES - 1) // (REQUESTS * MICROBATCHES))
snapshot_download(encoder.SOURCE.model_id, revision=encoder.SOURCE.revision,
                  allow_patterns=['config.json', 'tokenizer.json', 'model.safetensors'],
                  local_dir=str(MODEL))

## Import reusable training components


In [ ]:
import dataclasses
import functools
import hashlib
import json
from pathlib import Path

import jax
import numpy as np

from examples.magicbox import bundle as magicbox_bundle
from examples.magicbox import composition as magicbox
from examples.magicbox import data
from examples.magicbox import source
from minifield_training.batching import pointer as batching
from minifield_training.checkpoints import discovery
from minifield_training.checkpoints import training_state
from minifield_training.core import json_io
from minifield_training.engine import training_run
from minifield_training.evaluation import pointer as evaluate_pointer
from minifield_training.evaluation import schema_fields as evaluate
from minifield_training.models.lfm2_5 import encoder
from minifield_training.models.magicbox import pointer
from minifield_training.objectives import pointer as objective
from minifield_training.objectives import schema_fields as weighting
from minifield_training.optimizers import adamw
from minifield_training.optimizers import optax_adamw
from minifield_training.strategies import pretrained
from minifield_training.strategies import schema_fields as strategy

from minifield_training.supervisor import diagnostics

## Verify data and build the training plan


In [ ]:
devices = training_run.require_devices(DEVICES, 'tpu')
if (
    REQUESTS % DEVICES
    or EPOCHS < 1
    or VALIDATION_RECORDS < 1
    or FINAL_RECORDS < 0
):
    raise ValueError(
        "Invalid request topology, epoch count, or evaluation bound"
    )
mesh = (
    jax.sharding.Mesh(np.asarray(devices), ("data",))
    if DEVICES > 1
    else None
)
corpus = source.Corpus(
    DATASET, CACHE, allow_sample=False
)
config_path = MODEL / "config.json"
if json_io.digest_file(config_path) != encoder.SOURCE.config_sha256:
    raise ValueError("Pretrained config changed")
cfg = encoder.Adapter().parse_config(
    json_io.object_map(json.loads(config_path.read_text()))
)
head = pointer.Config(encoder_width=cfg.hidden_size)
if SEQUENCE_TOKENS is None or QUESTIONS is None:
    # Evaluation packs with the training shape, so every split must fit.
    measured_tokens, measured_questions = corpus.pointer_extent(
        sorted({str(shard["split"]) for shard in corpus.shards})
    )
    SEQUENCE_TOKENS = SEQUENCE_TOKENS or -(-measured_tokens // 128) * 128
    QUESTIONS = QUESTIONS or measured_questions
print('Joint tokens per request:', SEQUENCE_TOKENS, 'Questions per request:', QUESTIONS)
shape = batching.Shape(
    MICROBATCHES,
    REQUESTS,
    SEQUENCE_TOKENS,
    QUESTIONS,
    cfg.vocab_size,
    0,
)
if shape.sequence_tokens > encoder.MAX_SEQUENCE_LENGTH:
    raise ValueError("Requested tokens exceed the encoder context")
batches = batching.PointerBatchStrategy(shape, weighting.balance_types)
stream = source.training_stream(
    corpus, batches, SEED, EPOCHS,
    functools.partial(corpus.compile_pointer, split="train", score_width=SCORE_WIDTH),
)
inventory = magicbox.pointer_inventory(cfg, head)
print('Trainable parameters:', inventory.trainable_parameter_count, 'Frozen:', inventory.frozen_names)
optimizer = adamw.AdamWConfig(learning_rate=LEARNING_RATE)
TRANSACTION, OPTIMIZER_ID = (
    (optax_adamw.make_transaction, optax_adamw.implementation_identity(optimizer))
    if OPTIMIZER == 'optax'
    else (adamw.make_transaction, optimizer.implementation_identity)
)
identity = {
    "source": dataclasses.asdict(encoder.SOURCE),
    "encoder": dataclasses.asdict(cfg),
    "head": dataclasses.asdict(head),
    "shape": {
        key: value
        for key, value in dataclasses.asdict(shape).items()
        if key not in ("vocab_size", "pad_token_id")
    },
    "seed": SEED,
    "bf16": True,
    "devices": DEVICES,
    "contract": data.FORMAT,
    "template": data.POINTER_TEMPLATE,
    "score_width": SCORE_WIDTH,
    "implementation": "magicbox-pointer-jax/1",
}
source_id = hashlib.sha256(json_io.canonical(identity).encode()).hexdigest()
checkpoints = OUTPUT / "checkpoints"
resume = discovery.latest_checkpoint(
    checkpoints,
    run_id=RUN_ID,
    data_sha256=corpus.identity,
    source_id=source_id,
    reject_mismatched=True,
)

print('Resume:', resume, 'Updates per epoch:', stream.updates_per_epoch)
print(diagnostics.memory_snapshot())

## Load pretrained weights


In [ ]:
if resume is None:
    with diagnostics.monitor(OUTPUT / 'diagnostics'):
        _, parameters = pretrained.load_verified(MODEL, encoder.SOURCE, encoder.Adapter())
        parameters.update(pointer.initialize(head, jax.random.PRNGKey(SEED)))
    print('Loaded parameters:', sum(value.size for value in parameters.values()))
else:
    print('Weights will be restored with optimizer state in the next cell:', resume)

## Initialize or restore optimizer state


In [ ]:
if resume:
    current, cursor = training_state.load(
        resume,
        inventory,
        optimizer_id=OPTIMIZER_ID,
        run_id=RUN_ID,
        data_sha256=corpus.identity,
        source_id=source_id,
    )
else:
    current = adamw.initialize_state(parameters, inventory)
    del parameters
    cursor = training_state.Cursor(
        RUN_ID, corpus.identity, source_id, 0
    )
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / "run.json").write_text(
    json.dumps(
        {
            **identity,
            "optimizer": {"transaction": OPTIMIZER, **dataclasses.asdict(optimizer)},
            "data_sha256": corpus.identity,
            "source_id": source_id,
            "updates_per_epoch": stream.updates_per_epoch,
        },
        indent=2,
    )
)

print('Next update:', cursor.next_batch, diagnostics.memory_snapshot())

## Place state on TPU and inspect the first batch


In [ ]:
evaluator = evaluate.Evaluator(
    corpus.pointer_records,
    evaluate_pointer.Predictor(magicbox.bind_pointer(cfg, head, bf16=True), batches),
    names=data.KINDS,
    metrics=evaluate_pointer.Metrics,
)

update = strategy.make_step(
    magicbox.bind_pointer(cfg, head, bf16=True), inventory, optimizer, mesh=mesh,
    terms=objective.terms, transaction=TRANSACTION,
)
placement = (jax.sharding.NamedSharding(mesh, jax.sharding.PartitionSpec())
             if mesh is not None else jax.sharding.SingleDeviceSharding(devices[0]))
current = jax.device_put(current, placement)
jax.block_until_ready(current)
first_batches = stream(cursor.next_batch, None)
try:
    first_update = next(first_batches) if cursor.next_batch < EPOCHS * stream.updates_per_epoch else None
finally:
    first_batches.close()
physical_batch = ({key: value[0] for key, value in first_update.microbatches.items()}
                  if first_update is not None else None)
print('First physical batch:', {key: (value.shape, str(value.dtype))
                                for key, value in (physical_batch or {}).items()})
print(diagnostics.memory_snapshot())

## Lower the complete training step


In [ ]:
if first_update is not None:
    with diagnostics.monitor(OUTPUT / 'diagnostics'):
        print('Lowering the training step to compiler IR', flush=True)
        lowered_step = update.lower(current, first_update.microbatches, first_update.active)
    print('Training step lowering finished', flush=True)

## Compile the complete training step


In [ ]:
if first_update is not None:
    with diagnostics.monitor(OUTPUT / 'diagnostics'):
        print('Compiling the training step executable', flush=True)
        compiled_step = lowered_step.compile()
    print('Training step compilation finished', flush=True)
    print('Device memory:', compiled_step.memory_analysis(), flush=True)

## Execute the first 2 optimizer updates and save a checkpoint


In [ ]:
def report(event):
    line = json.dumps(event)
    print(line, flush=True)
    with (OUTPUT / 'progress.jsonl').open('a', encoding='utf-8') as log:
        log.write(line + '\n')

preflight_target = min(2, SMOKE_STEPS) if RUN_MODE == 'smoke' else 2
preflight_remaining = min(preflight_target, EPOCHS * stream.updates_per_epoch) - cursor.next_batch
if preflight_remaining > 0:
    with diagnostics.monitor(OUTPUT / 'diagnostics'):
        current, cursor = training_run.run(
            None, current, update, inventory,
            training_run.RunConfig(SEED, 2, 1, max_steps=preflight_remaining, max_seconds=3600, keep_checkpoints=KEEP_CHECKPOINTS),
            checkpoint_root=checkpoints, optimizer_id=OPTIMIZER_ID,
            cursor=cursor, report=report, required_platform='tpu', batch_source=stream,
            strict_compiles=True,
        )
print('Committed updates:', cursor.next_batch)
if cursor.next_batch > 0:
    training_state.verify_roundtrip(
        checkpoints / f'step-{cursor.next_batch:08d}', current, cursor, inventory,
        OPTIMIZER_ID,
    )
    print('Checkpoint parameters, moments, step, and cursor verified', flush=True)


## Validate the startup checkpoint


In [ ]:
with diagnostics.monitor(OUTPUT / 'diagnostics'):
    metrics = evaluator.run(current['params'], 'validation', 8)
print(json.dumps(metrics, indent=2))

## Continue the selected smoke or full run


In [ ]:
target = min(SMOKE_STEPS, EPOCHS * stream.updates_per_epoch) if RUN_MODE == 'smoke' else EPOCHS * stream.updates_per_epoch
remaining = target - cursor.next_batch
if remaining > 0:
    with diagnostics.monitor(OUTPUT / 'diagnostics'):
        current, cursor = training_run.run(
            None, current, update, inventory,
            training_run.RunConfig(SEED, 250, 10, max_steps=remaining, max_seconds=SESSION_HOURS * 3600, keep_checkpoints=KEEP_CHECKPOINTS),
            checkpoint_root=checkpoints, optimizer_id=OPTIMIZER_ID,
            cursor=cursor, evaluate=evaluator.callback(OUTPUT / 'metrics', VALIDATION_RECORDS),
            report=report, required_platform='tpu', batch_source=stream,
            strict_compiles=True,
        )
print('Committed updates:', cursor.next_batch)

## Evaluate and export the trained bundle


In [ ]:
completed = cursor.next_batch >= EPOCHS * stream.updates_per_epoch
if completed or RUN_MODE == 'smoke':
    for split in ("validation", "calibration", "test", "ood"):
        if any(shard["split"] == split for shard in corpus.shards):
            metrics = evaluator.run(
                current["params"], split, FINAL_RECORDS
            )
            (OUTPUT / f"final-{split}.json").write_text(
                json.dumps(metrics, indent=2)
            )
            print(json.dumps({"split": split, **metrics}), flush=True)
bundle = OUTPUT / f"bundle-{cursor.next_batch:08d}"
if not bundle.exists():
    magicbox_bundle.save_pointer(
        bundle,
        current["params"],
        cfg,
        head,
        encoder_config=config_path,
        tokenizer=DATASET / "tokenizer",
        step=cursor.next_batch,
    )
print(
    json.dumps(
        {
            "bundle": str(bundle),
            "completed_epochs": completed,
            "next_batch": cursor.next_batch,
        }
    ),
    flush=True,
)
size = sum(path.stat().st_size for path in OUTPUT.rglob('*') if path.is_file())
print(f'Output folder: {size / 1e9:.1f} GB (Kaggle saves up to 20 GB)', flush=True)


## Reload the exported bundle and predict


In [ ]:
from examples.magicbox import tokenizer

loaded_cfg, loaded_head, loaded_parameters, decode_config = magicbox_bundle.load_pointer(bundle)
mismatched = [
    name for name, value in loaded_parameters.items()
    if not np.array_equal(np.asarray(value), np.asarray(current['params'][name]))
]
if mismatched or set(loaded_parameters) != set(current['params']):
    raise RuntimeError(f'Reloaded bundle differs from the trained weights: {mismatched[:5]}')
print('Reloaded bundle matches all', len(loaded_parameters), 'trained tensors bit for bit', flush=True)
adapter = tokenizer.Adapter(bundle / 'tokenizer')
request = {
    'state': 'Ada is 37 years old and lives in London.',
    'questions': {'name': {'type': 'extract', 'instructions': 'Person name'}},
}
record = data.compile_pointer_record('inference', request, {}, adapter.encode)
predictor = evaluate_pointer.Predictor(
    magicbox.bind_pointer(loaded_cfg, loaded_head, bf16=True), batches,
    presence_threshold=float(str(decode_config['presence_threshold'])),
)
predictions, _ = predictor.score(loaded_parameters, record, include_losses=False)
print(json.dumps(data.format_results(record, predictions), indent=2))